The purpose of this code is to clean up our data and make it usable. It creates a pivot table so we can access the data easily and create new columns. Then it chooses the column we want, removes duplicates, and creates the columns we need so we can analyse such as met_performance and met_growth. 

In [ ]:
import pandas as pd
import numpy as np
import os, csv, re
import matplotlib.pyplot as plt
from pathlib import Path
plt.close("all")
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.0f}'.format) # Turn off scientific notation
pd.set_option('display.float_format', '{:.2f}'.format) # Force display floats with 2 decimal globally

def format_headers(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace(r"[./ >#:?*\-]", "_", regex=True)
        .str.replace(r"[()']", "", regex=True)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.lower()
        .str.replace(r"_+", "_", regex=True)
        .str.strip()
        .str.strip("_")
    )
    return df


from decimal import Decimal, ROUND_HALF_UP

In [ ]:
# Selects which data set we want to analyse

# subject="math"
subject="reading"
subject_title = subject.title()
subject_title

'Reading'

In [ ]:
# Opens the file that we want to use and read
df = pd.read_csv(f"exports/2526_diagnostic_results_{subject}_CONFIDENTIAL.csv")
df = format_headers(df)
df.columns

Index(['student_id', 'student_last_name', 'student_first_name',
       'student_grade', 'school', 'completion_date', 'norming_window',
       'most_recent_diagnostic_ytd_y_n', 'overall_relative_placement',
       'percent_progress_to_annual_typical_growth'],
      dtype='str')

In [ ]:
# Reduce dataframe to what we need

df = df[["student_id", "student_grade", "school", "norming_window", "most_recent_diagnostic_ytd_y_n",
          "overall_relative_placement", "percent_progress_to_annual_typical_growth"]]

In [ ]:
# Checks the value counts of our data

for col in df.columns:
    
    
    print(df[col].value_counts(dropna = False))
    print()

student_id
1000014    4
1000016    4
1000075    4
1000090    4
1000113    4
          ..
1006044    1
1006048    1
1006072    1
1006073    1
1006076    1
Name: count, Length: 6081, dtype: int64

student_grade
5     1342
9     1338
7     1328
10    1299
1     1278
K     1262
11    1256
3     1253
6     1243
4     1239
2     1238
12    1214
8     1173
Name: count, dtype: int64

school
East High                  1121
Falcon Ridge Elementary    1107
Cedar Hills Elementary     1095
Prairie Ridge Middle       1082
South High                 1052
Foothills Elementary       1051
Canyon Ridge Elementary    1047
Grand Mesa Elementary      1021
Eagle View Elementary      1013
Central High               1011
Bear Creek Elementary      1008
Mountain View Middle       1002
Heritage Middle             991
Aspen Elementary            970
Horizon Middle              947
North High                  945
Name: count, dtype: int64

norming_window
Spring (March 2 - End of Year)            6063
Winter (Novem

In [ ]:
# Creates a copy of our data frame

aoy_df = df.copy()

In [ ]:
# Check to see which grade levels exist

aoy_df["student_grade"].value_counts(dropna=False).sort_index()

student_grade
1     1278
10    1299
11    1256
12    1214
2     1238
3     1253
4     1239
5     1342
6     1243
7     1328
8     1173
9     1338
K     1262
Name: count, dtype: int64

In [ ]:
# If we have extra grades showing it gets rid of those extras

aoy_df = aoy_df[aoy_df["student_grade"].isin(["K","1","2","3","4","5","6","7","8"])]

aoy_df["student_grade"].value_counts(dropna=False).sort_index()

student_grade
1    1278
2    1238
3    1253
4    1239
5    1342
6    1243
7    1328
8    1173
K    1262
Name: count, dtype: int64

In [ ]:
# Creates met_performance column with base value 0 and assigns a 1 to all students that have met performance

aoy_df["met_performance"] = 0
aoy_df.loc[ (aoy_df["overall_relative_placement"] == "Mid or Above Grade Level") | 
    (aoy_df["overall_relative_placement"] == "Early On Grade Level"), "met_performance"] = 1

In [ ]:
# sorts our columns so later we can get rid of duplicates

aoy_df = aoy_df.sort_values("most_recent_diagnostic_ytd_y_n", ascending = False)
aoy_df

,student_id,student_grade,school,norming_window,most_recent_diagnostic_ytd_y_n,overall_relative_placement,percent_progress_to_annual_typical_growth,met_performance
16462,1006081,4,Cedar Hills Elementary,Spring (March 2 - End of Year),Y,Mid or Above Grade Level,206,1
12729,1004707,2,South High,Spring (March 2 - End of Year),Y,3 or More Grade Levels Below,77,0
5943,1002193,3,Heritage Middle,Spring (March 2 - End of Year),Y,3 or More Grade Levels Below,0,0
5942,1002192,7,Falcon Ridge Elementary,Spring (March 2 - End of Year),Y,2 Grade Levels Below,47,0
12732,1004708,2,Aspen Elementary,Spring (March 2 - End of Year),Y,1 Grade Level Below,0,0
...,...,...,...,...,...,...,...,...
6750,1002491,4,Falcon Ridge Elementary,Winter (November 16 - March 1),N,2 Grade Levels Below,258,0
6749,1002491,4,Falcon Ridge Elementary,Fall (Beginning of Year - November 15),N,Mid or Above Grade Level,705,1
6747,1002490,K,East High,Fall (Beginning of Year - November 15),N,Early On Grade Level,40,1
6742,1002488,3,Central High,Winter (November 16 - March 1),N,1 Grade Level Below,95,0


In [ ]:
# Gets rid of duplicates based on student_id and norming_window

aoy_df = aoy_df.drop_duplicates(subset = ["student_id", "norming_window"], keep = "first")

In [ ]:
# Fills in blanks with 0's in percent_progress_to_annual_typical_growth  

aoy_df = aoy_df.fillna({"percent_progress_to_annual_typical_growth" : 0})

In [ ]:
# Since NA is gone changes the type to an integer

aoy_df["percent_progress_to_annual_typical_growth"] =  aoy_df["percent_progress_to_annual_typical_growth"].astype(int) 

In [ ]:
# Creates a copy of just the end of year data

eoy_df = aoy_df[aoy_df["norming_window"] == "Spring (March 2 - End of Year)"].copy()

In [ ]:
# Creates a copy of just the beginning of year data

boy_df = aoy_df[aoy_df["norming_window"] == "Fall (Beginning of Year - November 15)"].copy()

In [ ]:
# Creates column in eoy_df, has_boy, and assigns a 1 if the student_id is in both eoy and boy and assigns a 0 if not

boy_students = set(boy_df["student_id"])

eoy_df["has_boy"] = np.where(eoy_df["student_id"].isin(boy_students), 1, 0)
eoy_df["has_boy"].value_counts(dropna=False)


has_boy
1    3354
0     643
Name: count, dtype: int64

In [ ]:
# If student_id is not in eoy and boy sets to blank. Then assigns values of 1 and 0 if the students growth is >= 100

eoy_df['met_growth'] = np.where(
    eoy_df['has_boy'] == 0,
    np.nan,
    np.where(eoy_df['percent_progress_to_annual_typical_growth'] >= 100, 1, 0)
)
eoy_df['met_growth'].value_counts(dropna=False)

met_growth
1.00    1756
0.00    1598
NaN      643
Name: count, dtype: int64

In [ ]:
# Creates a pivot table to sum and count the data

table = pd.pivot_table( eoy_df, values = ["met_performance", "met_growth"], index = ["school", "student_grade"], 
                        aggfunc = {"met_growth" : ["count","sum"], "met_performance" : ["count","sum"]})

In [ ]:
# Creates another pivot table with the same data but combined for the whole district

table2 = pd.pivot_table(eoy_df, values = ["met_performance", "met_growth"], index = "student_grade", 
                        aggfunc = {"met_growth" : ["count","sum"], "met_performance" : ["count","sum"]})
table2["school"] = "Summit Valley School District"
table2 = table2.set_index("school",append=True)
table2 = table2.swaplevel(0,1)

In [ ]:
# Calculates the met_growth percent for both tables

table["met_growth" , "percent"] = table["met_growth" , "sum"] / table["met_growth" , "count"]
table2["met_growth" , "percent"] = table2["met_growth" , "sum"] / table2["met_growth" , "count"]

In [ ]:
# Calculates the met_performance percent for both tables

table["met_performance", "percent"] = table["met_performance", "sum"] / table["met_performance", "count"]
table2["met_performance", "percent"] = table2["met_performance", "sum"] / table2["met_performance", "count"]

In [ ]:
# Puts the percent columns with the already existing met_growth and met_performance columns

table = table.sort_index(axis=1)
table2 = table2.sort_index(axis=1)

In [ ]:
# Concatenates together table and table2

table = pd.concat([table, table2])
table.head()

met_growth               met_performance  \
                                    count percent   sum           count   
school           student_grade                                            
Aspen Elementary 1                     23    0.52 12.00              28   
                 2                     20    0.35  7.00              23   
                 3                     24    0.46 11.00              25   
                 4                     26    0.42 11.00              34   
                 5                     20    0.40  8.00              24   

                                            
                               percent sum  
school           student_grade              
Aspen Elementary 1                0.57  16  
                 2                0.74  17  
                 3                0.48  12  
                 4                0.65  22  
                 5                0.58  14

In [ ]:
# Moves school index into df as a column

table = table.reset_index()
table.head()

index            school student_grade met_growth                \
                                             count percent   sum   
0     0  Aspen Elementary             1         23    0.52 12.00   
1     1  Aspen Elementary             2         20    0.35  7.00   
2     2  Aspen Elementary             3         24    0.46 11.00   
3     3  Aspen Elementary             4         26    0.42 11.00   
4     4  Aspen Elementary             5         20    0.40  8.00   

  met_performance              
            count percent sum  
0              28    0.57  16  
1              23    0.74  17  
2              25    0.48  12  
3              34    0.65  22  
4              24    0.58  14

In [ ]:
# Combines the stacked labels into 1. So met_growth and met_performance got divided into their percent, sum, and count

table.columns = [
    '_'.join(map(str, col)).strip()
    for col in table.columns
]
table.columns = table.columns.str.rstrip('_')

In [ ]:
# Creates a folder

output_dir = Path("intermediate")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Exports the table

table.to_csv(f"intermediate/01_CleanTest_{subject}.csv", index=False)
print("Done")

Done
